# سهم حسن قربان‌نژاد — آمار توصیفی داده‌های تشخیص ژانر موسیقی
این نوت‌بوک تعداد و سهم نمونه‌ها در هر سبک را از `data/processed/metadata.csv` حساب می‌کند و میانگین، میانه، واریانس و انحراف معیار تعداد نمونه‌ها را گزارش می‌دهد. تحلیل به فرادادهٔ مخزن محدود است؛ ویژگی عددی موج صوتی در این CSV موجود نیست.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# این نوت‌بوک را از ریشهٔ مخزن اجرا کنید.
metadata_path = Path("data/processed/metadata.csv")
if not metadata_path.exists():
    raise FileNotFoundError(f"فایل پیدا نشد: {metadata_path.resolve()}")
df = pd.read_csv(metadata_path)
required = {"file_name", "file_path", "genre"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"ستون‌های ضروری موجود نیستند: {sorted(missing)}")
df["genre"] = df["genre"].astype("string").str.strip().str.lower()
df = df.dropna(subset=["genre"])
print(f"تعداد رکوردهای برچسب‌دار: {len(df)}")
print(f"تعداد سبک‌ها: {df['genre'].nunique()}")

In [ ]:
counts = df["genre"].value_counts().sort_index().rename_axis("genre").to_frame("count")
counts["percentage"] = counts["count"] / counts["count"].sum() * 100
display(counts.round({"percentage": 2}))

In [ ]:
genre_counts = counts["count"].astype(float)
summary = pd.Series({
    "total_files": genre_counts.sum(),
    "number_of_genres": genre_counts.size,
    "mean_files_per_genre": genre_counts.mean(),
    "median_files_per_genre": genre_counts.median(),
    "population_variance_ddof0": genre_counts.var(ddof=0),
    "sample_variance_ddof1": genre_counts.var(ddof=1),
    "population_std_ddof0": genre_counts.std(ddof=0),
    "minimum": genre_counts.min(),
    "maximum": genre_counts.max(),
}, name="value")
display(summary.to_frame().round(3))
print("یادآوری: واریانس جامعه توصیف همین سبک‌های ثبت‌شده است؛ واریانس نمونه از ddof=1 استفاده می‌کند.")

In [ ]:
ax = counts["count"].plot(kind="bar", figsize=(8, 4.5), color="#5267d8")
ax.set_title(f"Audio files per genre (n={int(genre_counts.sum())})")
ax.set_xlabel("Genre")
ax.set_ylabel("Number of files")
for bar, pct in zip(ax.patches, counts["percentage"]):
    ax.annotate(f"{int(bar.get_height())} ({pct:.1f}%)",
                (bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha="center", va="bottom", xytext=(0, 3), textcoords="offset points")
ax.set_ylim(0, genre_counts.max() + 1.5)
ax.grid(axis="y", alpha=.25)
plt.tight_layout()
plt.show()

## تفسیر
در نسخه‌ای از metadata مخزن که ۱۴ فایل در پنج سبک دارد، شمارش‌ها عبارت‌اند از Blues=3، Classical=2، Hip-hop=5، Jazz=1 و Rock=3. بنابراین میانگین شمارش ۲٫۸، واریانس جامعه ۱٫۷۶ و واریانس نمونه ۲٫۲۰ است. نمونه‌ها کم و نامتوازن‌اند؛ به‌خصوص از یک نمونهٔ Jazz نمی‌توان ارزیابی مدل قابل‌اتکا انجام داد. اعداد را با اجرای سلول‌ها روی نسخهٔ فعلی CSV بازبینی کنید.